## Tool to find what to search in the butler

- reference : https://github.com/lsst/tutorial-notebooks/blob/main/DP1/100_How_to_Use_RSP_Tools/104_Butler_data_access/104_2_Explore_the_butler_repo.ipynb

---
- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab/IN2P3/CNRS, Université Paris-Saclay
- **Created:** 2026-06-26
- **Last update:** 2026-06-26


In [ ]:
from lsst.daf.butler import Butler
import lsst.sphgeom as sphgeom
import lsst.geom as geom
from lsst.utils.plotting import (
    get_multiband_plot_colors,
    get_multiband_plot_symbols,
    get_multiband_plot_linestyles,
)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib.patches import Polygon

## 1 Configuration

In [ ]:
# ── Butler ────────────────────────────────────────────────────────────────────
repo = "dp2_prep"

collection = [
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage4",
]

instrument = "LSSTCam"
skymapName = "lsst_cells_v2"

# ── Input targets ─────────────────────────────────────────────────────────────
target_file = "summary_visit_counts_per_star_V17-21_r2.0deg.csv"

# ── Cross-match search radius ─────────────────────────────────────────────────
MATCH_RADIUS_ARCSEC = 1.0  # maximum separation for a valid match [arcsec]


print("Butler configuration done.")

In [ ]:
butler = Butler(repo, collections=collection)

assert butler is not None


registry = butler.registry
skymap = butler.get("skyMap", skymap=skymapName, collections=collection)
print(f"Butler initialised | repo: {repo}")

## 2. Explore the Butler

### 2.1. Repositories and collections

In [ ]:
butler.collections.query_info("LSSTCam/runs/DRP/DP2")

In [ ]:
print("Butler repository : ", butler.get_known_repos())

In [ ]:
all_collections = butler.collections.query("*")

In [ ]:
# for collection in all_collections:
#    if "calib" in collection:
#         print(collection)

In [ ]:
del all_collections

### 2.2. Dataset types

In [ ]:
all_dt = list(butler.registry.queryDatasetTypes("*object*"))
for dt in all_dt:
    if dt.storageClass_name == "ArrowTable" or "Arrow" in dt.storageClass_name:
        print(dt)
del all_dt

In [ ]:
all_dt = list(butler.registry.queryDatasetTypes("*source*"))
for dt in all_dt:
    if dt.storageClass_name == "ArrowTable" or "Arrow" in dt.storageClass_name:
        print(dt)
del all_dt

In [ ]:
### 2.3. Dimensions and elements

In [ ]:
butler.get_dataset_type("source")

## 3. Visualize Butler metadata

### 3.1. Tract and patch boundaries


In [ ]:
region = sphgeom.Region.from_ivoa_pos("CIRCLE 53.076 -28.110 10.0")
tract_dimrecs = butler.query_dimension_records(
    "tract", where="tract.region OVERLAPS :region", bind={"region": region}
)
patch_dimrecs = butler.query_dimension_records(
    "patch", where="patch.region OVERLAPS :region", bind={"region": region}
)
print("Number of overlapping patches: ", len(patch_dimrecs))

### 3.2. Visit dates

In [ ]:
region = sphgeom.Region.from_ivoa_pos("CIRCLE 53.076 -28.110 10.0")